# 🧩 Middleware: Controlling the Agent Loop

## Learning Objectives
In this notebook, you will learn:
1. **What middleware is** - hooks that run inside the agent loop, around the model and tool calls
2. **`SummarizationMiddleware`** - compress old history automatically, triggered by messages, tokens, or context fraction
3. **`HumanInTheLoopMiddleware`** - pause before a risky tool call for approval, editing, or rejection
4. **Checkpointers and `thread_id`** - the persistence that makes both of the above possible
5. **Resuming with `Command`** - how an interrupted agent picks up where it left off
6. **`PIIMiddleware`, `ModelCallLimitMiddleware`, and writing your own** - built-in guardrails plus a custom `AgentMiddleware` subclass

## Prerequisites
- Completed `7.0_LangChain_First_Agent.ipynb` through `7.4_Structured_Output.ipynb`
- `pip install langchain langchain-openai langgraph python-dotenv`
- A `.env` file with `OPENAI_API_KEY`

---
## 💡 Part 1: What Is Middleware?

Middleware gives you tighter control over what happens **inside** the agent loop. Rather than
rebuilding the loop as raw LangGraph nodes, you attach hooks that run at defined points — before
the model, after the model, around a tool call.

Middleware is useful for:

- **Observability** — logging, analytics, and debugging agent behaviour
- **Transformation** — rewriting prompts, filtering tool selection, formatting output
- **Control flow** — retries, fallbacks, and early termination
- **Guardrails** — rate limits, PII detection, human approval

### Key Insight:
Every built-in middleware compiles down to extra nodes in the agent's graph. Run
`agent.get_graph().nodes` after building one and you will see entries like
`SummarizationMiddleware.before_model` — nothing magical is happening, the hooks are just
graph nodes that `create_agent` wires in for you.

---
## 🔑 Part 2: Environment Setup

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API credentials from .env
# ============================================================================
import os

from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "❌ Set OPENAI_API_KEY in your .env file"
print("✅ Environment loaded successfully!")

---
## 🗜️ Part 3: Summarization Middleware

`SummarizationMiddleware` automatically summarizes conversation history as it approaches a
threshold, compressing older context while preserving recent messages verbatim.

It is useful for:
- Long-running conversations that would otherwise exceed the context window
- Multi-turn dialogues with extensive history
- Applications where preserving the *gist* of full context matters more than the exact wording

### Key Concepts:
- **`model`** — required. The summarizer needs its own model to write the summary with; it can
  be cheaper than your main agent model.
- **`trigger`** — *when* to summarize. Three units: `("messages", n)`, `("tokens", n)`, or
  `("fraction", f)` of the model's context window.
- **`keep`** — *how much* recent context survives untouched, in the same three units.
- **A checkpointer is required.** Summarization rewrites history between turns, so the agent
  needs somewhere to persist that history — hence `InMemorySaver()` and a `thread_id`.

### 3.1 📨 Trigger by Message Count

The most predictable trigger: summarize once the conversation exceeds 10 messages, keeping the
4 most recent. Easy to reason about, but message counts are a poor proxy for context pressure
when messages vary wildly in size.

In [ ]:
# ============================================================================
# SUMMARIZATION: Message-count trigger
# ============================================================================
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="gpt-4o-mini",
    checkpointer=InMemorySaver(),   # required: history must persist between turns
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",        # the model that WRITES the summary
            trigger=("messages", 10),   # summarize once history exceeds 10 messages
            keep=("messages", 4),       # keep the 4 most recent verbatim
        )
    ],
)

print("🧩 Graph nodes:", list(agent.get_graph().nodes))

In [ ]:
agent

#### 🧵 Threads and `thread_id`

The checkpointer stores state per **thread**. Passing the same `thread_id` on every call is
what makes these separate `.invoke()` calls one continuous conversation — change the id and you
start fresh.

In [ ]:
# ============================================================================
# THREAD CONFIG: Same thread_id = one continuous conversation
# ============================================================================
config = {"configurable": {"thread_id": "test-1"}}

print("🧵 Using thread:", config["configurable"]["thread_id"])

In [ ]:
# ============================================================================
# RUN: Watch the message count grow, then collapse when summarization fires
# ============================================================================
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    # The count DROPS on the turn where summarization kicks in.
    print(f"❓ {q:<16} -> 📋 {len(response['messages'])} messages in history")

### 3.2 🔢 Trigger by Token Count

Tokens are the resource that actually runs out, so a token trigger tracks context pressure far
better than a message count. The thresholds here are set deliberately low (550 / 200) so
summarization fires within a few turns instead of after hundreds.

The hotel tool returns a deliberately verbose string — that is how we burn tokens quickly
enough to see the behaviour in a short demo.

In [ ]:
# ============================================================================
# SUMMARIZATION: Token-count trigger
# ============================================================================
from langchain.tools import tool


@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 550),   # low on purpose, so the demo triggers fast
            keep=("tokens", 200),
        ),
    ],
)

config = {"configurable": {"thread_id": "tokens-demo"}}


def count_tokens(messages) -> int:
    """Rough token estimate for display only (~4 chars per token)."""
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4


print("✅ Token-triggered agent ready")

In [ ]:
# ============================================================================
# RUN: Token count climbs, then drops when the summary replaces old turns
# ============================================================================
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config,
    )
    print(f"🌍 {city:<10} ~{count_tokens(response['messages']):>5} tokens, "
          f"{len(response['messages'])} messages")

### 3.3 📐 Trigger by Context Fraction

The most portable trigger: a **fraction of the model's context window**. Swap `gpt-4o-mini`
(128k) for a model with a 1M-token window and `("fraction", 0.7)` keeps meaning the same
thing, while a hard-coded token count would not.

The fractions below are absurdly small (0.5% and 0.2%) purely so the demo triggers in a handful
of turns. In production you would use something like `("fraction", 0.7)` / `("fraction", 0.3)`.

In [ ]:
# ============================================================================
# SUMMARIZATION: Context-fraction trigger
# ============================================================================
@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"


agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("fraction", 0.005),  # 0.5% of 128k ≈ 640 tokens — tiny, for the demo
            keep=("fraction", 0.002),     # 0.2% ≈ 256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "fraction-demo"}}

CONTEXT_WINDOW = 128_000  # gpt-4o-mini

cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config,
    )
    tokens = count_tokens(response["messages"])
    print(f"🌍 {city:<10} ~{tokens:>5} tokens ({tokens / CONTEXT_WINDOW:.4%}), "
          f"{len(response['messages'])} msgs")

---
## 🙋 Part 4: Human-in-the-Loop Middleware

`HumanInTheLoopMiddleware` pauses agent execution for human approval, editing, or rejection of
tool calls **before** they execute.

It is useful for:
- High-stakes operations requiring sign-off (database writes, financial transactions)
- Compliance workflows where human oversight is mandatory
- Long-running conversations where human feedback steers the agent

### Key Concepts:
- **`interrupt_on`** maps tool name → policy. A dict with `allowed_decisions` gates that tool;
  `False` lets it run untouched. Gate only what is dangerous — pausing on reads is friction
  with no safety benefit.
- **The pause is a real suspension.** The agent stops, its state is checkpointed, and
  `result` contains an `"__interrupt__"` key. Nothing runs until you resume.
- **Resume with `Command(resume=...)`**, passing a `decisions` list — one decision per gated
  call.

In [ ]:
# ============================================================================
# HITL SETUP: Tools + one gated agent, reused across all three scenarios
# ============================================================================
import uuid

from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"


def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"


agent = create_agent(
    model="gpt-4o",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),   # required: the pause must persist
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                # Sending is irreversible -> require a human decision.
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                # Reading is harmless -> let it run without interruption.
                "read_email_tool": False,
            }
        )
    ],
)


def new_thread(label: str) -> dict:
    """Build a config on a FRESH thread id.

    Re-running a Step 1 cell on a thread that is already parked at an interrupt
    is what triggers the OpenAI 400 described below, so every Step 1 gets a
    brand-new thread.
    """
    return {"configurable": {"thread_id": f"{label}-{uuid.uuid4().hex[:8]}"}}


print("🙋 HITL agent ready — send_email_tool is gated, read_email_tool is not")

> **Note**: one agent is enough for all three scenarios below. Each uses its own `thread_id`,
> which gives it an isolated conversation — no need to rebuild the agent per case.

### ⚠️ The Pitfall: Never Re-run a Step 1 Cell on a Parked Thread

When the middleware interrupts, `after_model` has **already written** the `AIMessage` carrying
the `tool_calls` into the checkpointed state. That thread is now parked mid-tool-call, and the
only valid way forward is `Command(resume=...)`.

If you instead invoke the agent again with a *new* user message on the same `thread_id` — which
is exactly what happens when you re-run a Step 1 cell — the pending interrupt is discarded and
your message is appended. History becomes:

```
[0] HumanMessage      "Send email to ..."
[1] AIMessage         tool_calls=[send_email_tool]   <- never answered
[2] HumanMessage      "Send email to ..."            <- should have been a ToolMessage
```

The model node then fires and the provider rejects the request:

```
BadRequestError: Error code: 400 - An assistant message with 'tool_calls' must be
followed by tool messages responding to each 'tool_call_id'. The following
tool_call_ids did not have response messages: call_...
(param: 'messages.[2].role')
```

`param` points at index 2 — the slot where a `ToolMessage` was required.

**The fix**: give every Step 1 a fresh thread. `new_thread()` above does that with a random
suffix, so re-running any cell below is always safe. If you do strand a thread, you do not need
to rebuild the agent — just start a new thread id.

In [ ]:
agent

### 4.1 ✅ Scenario: Approve

Step 1 requests the send. The agent runs up to the gated tool and **stops**, returning an
`"__interrupt__"` key instead of a final answer.

In [ ]:
# ============================================================================
# APPROVE - STEP 1: Request, and hit the interrupt
# ============================================================================
config = new_thread("approve")   # fresh thread -> safe to re-run this cell

result = agent.invoke(
    {"messages": [HumanMessage(
        content="Send email to john@test.com with subject 'Hello' and body 'How are you?'"
    )]},
    config=config,
)

print(f"🧵 thread: {config['configurable']['thread_id']}")
print("⏸️ Interrupted!" if "__interrupt__" in result else "▶️ Ran to completion")
result

In [ ]:
# ============================================================================
# APPROVE - STEP 2: Resume with an approve decision
# ============================================================================
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

    result = agent.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}),
        config=config,
    )

    print(f"✅ Result: {result['messages'][-1].content}")
else:
    print("⚠️ No pending interrupt on this thread — re-run Step 1 first.")

In [ ]:
# ============================================================================
# INSPECT: Full state after approval — the tool did execute
# ============================================================================
result

### 4.2 ❌ Scenario: Reject

Same request, a different decision. On `reject` the tool **never executes**; the agent receives
a message saying the call was refused and continues from there — typically by explaining to the
user that it could not send.

In [ ]:
# ============================================================================
# REJECT - STEP 1: Same request on its own fresh thread
# ============================================================================
config = new_thread("reject")

result = agent.invoke(
    {"messages": [HumanMessage(
        content="Send email to john@test.com with subject 'Hello' and body 'How are you?'"
    )]},
    config=config,
)

print(f"🧵 thread: {config['configurable']['thread_id']}")
print("⏸️ Interrupted!" if "__interrupt__" in result else "▶️ Ran to completion")

In [ ]:
# ============================================================================
# REJECT - STEP 2: Resume with a reject decision
# ============================================================================
if "__interrupt__" in result:
    print("⏸️ Paused! Rejecting...")

    result = agent.invoke(
        Command(resume={"decisions": [{"type": "reject"}]}),
        config=config,
    )

    print(f"❌ Result: {result['messages'][-1].content}")
else:
    print("⚠️ No pending interrupt on this thread — re-run Step 1 first.")

In [ ]:
# ============================================================================
# INSPECT: Full state after rejection — no email was sent
# ============================================================================
result

### 4.3 ✏️ Scenario: Edit

The most powerful decision: **rewrite the arguments** before the tool runs. Here the model
picks the wrong recipient, and the human corrects it in flight.

`edited_action` carries the tool `name` plus a complete replacement `args` dict — the tool then
executes with your values, not the model's.

In [ ]:
# ============================================================================
# EDIT - STEP 1: Request with deliberately wrong information
# ============================================================================
config = new_thread("edit")

result = agent.invoke(
    {"messages": [HumanMessage(
        content="Send email to wrong@email.com with subject 'Test' and body 'Hello'"
    )]},
    config=config,
)

print(f"🧵 thread: {config['configurable']['thread_id']}")
print("⏸️ Interrupted!" if "__interrupt__" in result else "▶️ Ran to completion")
result

In [ ]:
# ============================================================================
# EDIT - STEP 2: Correct the arguments, then let the tool run
# ============================================================================
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",   # the tool to run
                            "args": {                    # replacement arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending",
                            },
                        },
                    }
                ]
            }
        ),
        config=config,
    )

    print(f"✏️ Result: {result['messages'][-1].content}")
else:
    print("⚠️ No pending interrupt on this thread — re-run Step 1 first.")

In [ ]:
# ============================================================================
# INSPECT: Full state after the edit — note the corrected recipient
# ============================================================================
result

---
## 🧰 Part 5: More Built-in Middleware, and Writing Your Own

The middleware above (`SummarizationMiddleware`, `HumanInTheLoopMiddleware`) are two of
several built-ins. This section adds `PIIMiddleware` and `ModelCallLimitMiddleware`, then
shows how to write your **own** middleware and stack several together.


In [ ]:
# ============================================================================
# SETUP: A second agent, plus a tool for the built-in/custom middleware demos
# ============================================================================
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)


@tool
def lookup_account(account_id: str) -> str:
    """Look up a fake account record by id."""
    return f"Account {account_id}: balance=₹42,500, status=active, email=user{account_id}@example.com"


### 5.1 Built-in middleware: `PIIMiddleware`, `ModelCallLimitMiddleware`

- `PIIMiddleware` — redacts sensitive fields (emails, phone numbers, etc.) before they reach
  the model or the log
- `ModelCallLimitMiddleware` / `ToolCallLimitMiddleware` — cost / runaway-loop guards

> Import paths can shift between minor releases — if an import below fails, check
> `from langchain.agents.middleware import ...` against your installed version with
> `pip show langchain`.


In [ ]:
# ============================================================================
# BUILT-IN MIDDLEWARE: PII redaction + a model-call limit
# ============================================================================
from langchain.agents.middleware import (
    ModelCallLimitMiddleware,
    PIIMiddleware,
)

agent_with_builtins = create_agent(
    model=llm,
    tools=[lookup_account],
    system_prompt="You are a banking support assistant.",
    middleware=[
        # One PIIMiddleware per PII type - the type is the first positional
        # arg, not a `pii_types` list. Built-ins: email, credit_card, ip,
        # mac_address, url (or pass your own regex/callable via `detector`).
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        # Caps model calls: `run_limit` per invocation, `thread_limit`
        # across a whole conversation thread.
        ModelCallLimitMiddleware(run_limit=5),
    ],
)

# Do NOT end a cell with a bare `agent_with_builtins`. Jupyter would call the
# object's `_repr_mimebundle_`, which calls `draw_mermaid_png()` -> POSTs the
# diagram to the mermaid.ink web service -> ValueError when it's unreachable.
# Printing the graph structure is local and needs no network:
print(type(agent_with_builtins).__name__)
print("Nodes:", list(agent_with_builtins.get_graph().nodes))


In [ ]:
result = agent_with_builtins.invoke({
    "messages": [{"role": "user", "content": "Look up account 1029 and tell me the balance."}]
})

for m in result["messages"]:
    print(f"[{m.type}] {m.content}")


### 5.2 Writing your own middleware

Custom middleware subclasses `AgentMiddleware` and overrides the hook for the point in
the loop you want to intercept. Two rules matter and are easy to get wrong:

1. **Every hook takes `(self, state, runtime)`.** The `runtime` argument is not
   optional — a hook defined as `before_model(self, state)` raises `TypeError`
   at run time.
2. **Hooks return a state *update*, they do not mutate state.** `state` is a
   plain dict snapshot handed to your node; assigning `state["x"] = ...` and
   returning `None` throws the write away. To pass a value from `before_model`
   to `after_model` you must (a) declare a channel for it on a custom state
   schema, and (b) `return {"x": ...}`.

The timing logger below does both. `TimingState` extends `AgentState` with one
`NotRequired` key, and `state_schema` tells the agent to merge that channel into
its own state.


In [ ]:
# ============================================================================
# A CUSTOM MIDDLEWARE: log latency around each model call
# ============================================================================
import time

from typing_extensions import NotRequired

from langchain.agents.middleware import AgentMiddleware, AgentState


class TimingState(AgentState):
    """AgentState plus one extra channel for the timer handoff."""

    model_call_started_at: NotRequired[float]


class TimingLoggerMiddleware(AgentMiddleware[TimingState]):
    """Logs latency around each model call in the agent loop."""

    state_schema = TimingState

    def before_model(self, state, runtime):
        print(">> calling model...")
        return {"model_call_started_at": time.time()}  # a state UPDATE

    def after_model(self, state, runtime):
        started = state.get("model_call_started_at")
        if started is not None:
            print(f"<< model responded in {time.time() - started:.2f}s")
        return None  # nothing to write back


# Note: for timing specifically, the `wrap_model_call` hook is simpler still --
# it brackets the call in a single method, so no state channel is needed at all.
# We use before/after here because it generalizes to hooks that must inspect
# state between steps.


In [ ]:
agent_with_logging = create_agent(
    model=llm,
    tools=[lookup_account],
    system_prompt="You are a banking support assistant.",
    middleware=[TimingLoggerMiddleware()],
)

result = agent_with_logging.invoke({
    "messages": [{"role": "user", "content": "What's the status of account 1029?"}]
})
print("\nFinal answer:", result["messages"][-1].content)


### 5.3 Stacking middleware

Middlewares run in the order you list them. A typical production stack mirrors what
you'd hand-roll as custom LangGraph nodes:


In [ ]:
# ============================================================================
# COMPOSING BUILT-IN + CUSTOM MIDDLEWARE
# ============================================================================
production_agent = create_agent(
    model=llm,
    tools=[lookup_account],
    system_prompt="You are a banking support assistant.",
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        ModelCallLimitMiddleware(run_limit=8),
        TimingLoggerMiddleware(),
    ],
)

# `create_agent` returns a compiled LangGraph, not a wrapper object - there is
# no `.middleware` attribute on it. Inspect the graph instead:
print(type(production_agent).__name__)
print("Nodes:", list(production_agent.get_graph().nodes))


---
## 📝 Summary

In this notebook, we learned:

### 1. What Middleware Does
- **Hooks inside the agent loop**: observability, prompt/tool transformation, control flow,
  and guardrails
- **They are just graph nodes**: `agent.get_graph().nodes` shows each middleware's hooks wired
  into the compiled graph

### 2. `SummarizationMiddleware`
- **Needs its own `model`** to write summaries with — often a cheaper one than the agent uses
- **Three trigger units**: `("messages", n)` is predictable, `("tokens", n)` tracks the real
  constraint, `("fraction", f)` stays correct when you change models
- **`keep`** controls how much recent context survives verbatim
- **Requires a checkpointer**, because it rewrites history between turns

### 3. `HumanInTheLoopMiddleware`
- **`interrupt_on`** gates per tool: a policy dict to require a decision, `False` to allow
  freely — gate irreversible actions, not reads
- **Three decisions**: `approve` runs it, `reject` blocks it, `edit` rewrites the arguments first
- **`"__interrupt__"` in the result** signals a real suspension, not an error

### 4. Threads and Resumption
- **`thread_id`** isolates conversations; one agent serves many independent threads
- **`Command(resume={"decisions": [...]})`** picks up exactly where the interrupt paused,
  with one decision per gated call
- **Never re-invoke a parked thread with a new message**: the unanswered `AIMessage`
  tool call is already checkpointed, so the next model call fails with a 400 —
  "tool_call_ids did not have response messages". Resume it, or start a new thread

### Next Steps
- Explore the remaining built-ins — `PIIMiddleware`, `ModelCallLimitMiddleware`,
  `ToolCallLimitMiddleware` — and write a custom `AgentMiddleware` subclass with
  `before_model` / `after_model` / `wrap_model_call` hooks